In [ ]:
import numpy as np
import pandas as pd
from decoding_core.utils import get_all_paths

In [4]:
weights_paths = get_all_paths("/Users/saru/Local_Work/neuromodul/fscv/tests_code/data_1d_vxlbl",collapsed=False)
voltammograms_path = "/Users/saru/Local_Work/neuromodul/fscv/tests_code/data_1d_vxlbl/AFOR/bfa/155Vs/10Hz/2025_07_07__AFOR__DA_5HT_NE__155Vs_10Hz__BFA_INM001_99bW06R09/DA_5HT_NE/voltammograms.npy"
labels_path = "/Users/saru/Local_Work/neuromodul/fscv/tests_code/data_1d_vxlbl/AFOR/bfa/155Vs/10Hz/2025_07_07__AFOR__DA_5HT_NE__155Vs_10Hz__BFA_INM001_99bW06R09/DA_5HT_NE/labels.npy"


In [ ]:
from decoding_core.utils import load_activation_data
train, val, test, y_mean, y_std = load_activation_data(weights_paths, batch_size=32,collapsed=False)


Getting pairs not collapsed.
First activation shape(2250, 1000, 80)
First label shape(2250, 4)
Expert activation set shape: (9750, 1000, 80)
Labels shape: (9750, 4)
First label row (all 4 cols): [0.  0.  0.  7.4]
y_mean: [771.21655 246.0114  260.2792 ]  y_std: [749.0034  448.34094 460.35495]


In [ ]:
from decoding_core.models import Regressor, train_regressor, test_regressor
from torch.optim import Adam
from torch import nn

In [5]:
loss = nn.MSELoss()

In [6]:
losses = {}
modes = ['attention', 'mean', 'none']

for mode in modes:
    print("----- Mode: ", mode,"--------")
    model = Regressor(80, 3, pooling=mode)
    optim = Adam(model.parameters(), lr=1e-3)
    train_loss = train_regressor(model=model,train_data_loader=train, val_loader=val,loss=loss,optimizer=optim, device="mps", num_epochs=100)

    losses[mode+" train"] = train_loss
    test_loss = test_regressor(model=model,test_data_loader=test,loss=loss,y_mean=y_mean,y_std=y_std, device='mps')
    losses[mode+" test"] = test_loss


----- Mode:  attention --------
Epoch 10/100| Train Loss: 0.7916| Val Loss: 0.9653
Epoch 20/100| Train Loss: 0.7692| Val Loss: 1.0135
Epoch 30/100| Train Loss: 0.7551| Val Loss: 1.0198
Epoch 40/100| Train Loss: 0.7492| Val Loss: 1.0253
Epoch 50/100| Train Loss: 0.7473| Val Loss: 1.0033
Epoch 60/100| Train Loss: 0.7487| Val Loss: 1.0090
Epoch 70/100| Train Loss: 0.7473| Val Loss: 1.0046
Epoch 80/100| Train Loss: 0.7475| Val Loss: 1.0039
Epoch 90/100| Train Loss: 0.7446| Val Loss: 1.0060
Epoch 100/100| Train Loss: 0.7460| Val Loss: 1.0051
----- Mode:  mean --------
Epoch 10/100| Train Loss: 0.8474| Val Loss: 0.9696
Epoch 20/100| Train Loss: 0.8254| Val Loss: 0.9275
Epoch 30/100| Train Loss: 0.7871| Val Loss: 0.9139
Epoch 40/100| Train Loss: 0.7686| Val Loss: 0.9477
Epoch 50/100| Train Loss: 0.7593| Val Loss: 0.9275
Epoch 60/100| Train Loss: 0.7604| Val Loss: 0.9045
Epoch 70/100| Train Loss: 0.7549| Val Loss: 0.9222
Epoch 80/100| Train Loss: 0.7522| Val Loss: 0.9031
Epoch 90/100| Train Lo

In [7]:
losses.keys()

dict_keys(['attention train', 'attention test', 'mean train', 'mean test', 'none train', 'none test'])

In [8]:
for name, loss in losses.items():
    if "test" in name:
        print("-----",name,"-----\n",loss)

----- attention test -----
 (2.822792407060281, tensor([0.6047, 1.2730, 1.3164]), tensor([464.5112, 478.7674, 495.1039]), tensor([[ 348.6577,  503.0963,  272.0315],
        [ 487.3418,  394.6113,  436.6172],
        [ 371.9158,  343.0250,  441.6282],
        ...,
        [1087.8143,   73.3763,   73.5171],
        [ 453.9796,  372.7557,  416.3387],
        [1157.9648,   12.3446,   12.6481]]), tensor([[6.1035e-05, 1.5259e-05, 1.5000e+03],
        [6.1035e-05, 1.0000e+03, 0.0000e+00],
        [5.0000e+02, 1.5259e-05, 5.0000e+02],
        ...,
        [1.0800e+03, 1.5259e-05, 0.0000e+00],
        [5.0000e+02, 1.5259e-05, 5.0000e+02],
        [6.1035e-05, 8.4000e+02, 1.6800e+03]]))
----- mean test -----
 (2.6522131516383243, tensor([0.5881, 1.2375, 1.2878]), tensor([451.7775, 465.4258, 484.3391]), tensor([[ 892.9375,  195.8520,  101.9208],
        [1011.8051,   88.6351,   81.1592],
        [1118.6055,   52.0426,   19.8467],
        ...,
        [1100.3695,   69.5680,   35.2132],
        [ 4